In [2]:
!pip install openai

In [20]:
import pandas as pd
import requests

In [4]:
kpi_summary = pd.read_csv("kpi_summary.csv")

industry_summary = pd.read_csv("industry_summary.csv")

billing_summary = pd.read_csv("billing_summary.csv")

feature_summary = pd.read_csv("feature_summary.csv")

priority_summary = pd.read_csv("priority_summary.csv")

churn_reason_summary = pd.read_csv("churn_reason_summary.csv")

In [21]:
url = "https://open.er-api.com/v6/latest/USD"

response = requests.get(url)

exchange = response.json()

In [22]:
exchange.keys()

dict_keys(['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates'])

In [23]:
usd_to_inr = exchange["rates"]["INR"]
usd_to_eur = exchange["rates"]["EUR"]
usd_to_gbp = exchange["rates"]["GBP"]

print(usd_to_inr)
print(usd_to_eur)
print(usd_to_gbp)

96.611711
0.876364
0.74772


In [26]:
billing_summary["total_mrr_inr"] = billing_summary["Total_MRR"] * usd_to_inr
billing_summary["average_mrr_inr"] = billing_summary["Average_MRR"] * usd_to_inr

billing_summary["total_mrr_eur"] = billing_summary["Total_MRR"] * usd_to_eur
billing_summary["average_mrr_eur"] = billing_summary["Average_MRR"] * usd_to_eur

billing_summary["total_mrr_gbp"] = billing_summary["Total_MRR"] * usd_to_gbp
billing_summary["average_mrr_gbp"] = billing_summary["Average_MRR"] * usd_to_gbp

display(billing_summary)

,billing_frequency,Total_Subscriptions,Total_MRR,Average_MRR,total_mrr_inr,average_mrr_inr,total_mrr_eur,average_mrr_eur,total_mrr_gbp,average_mrr_gbp
0,annual,2461,5597398,27293.285656,5.407742e+08,2.636851e+06,4.905358e+06,23918.852991,4.185286e+06,20407.735551
1,monthly,2539,5741349,27135.166601,5.546816e+08,2.621575e+06,5.031512e+06,23780.283143,4.292921e+06,20289.506771


In [33]:
accounts = pd.read_csv("accounts_clean.csv")
subscriptions = pd.read_csv("subscriptions_clean.csv")
feature_usage = pd.read_csv("feature_usage_clean.csv")
support_tickets = pd.read_csv("support_ticket_clean.csv")
churn_events = pd.read_csv("churn_events_clean.csv")

In [29]:
subscriptions["mrr_inr"] = subscriptions["mrr_amount"] * usd_to_inr

subscriptions["mrr_eur"] = subscriptions["mrr_amount"] * usd_to_eur

subscriptions["mrr_gbp"] = subscriptions["mrr_amount"] * usd_to_gbp


In [34]:
subscriptions["arr_inr"] = subscriptions["arr_amount"] * usd_to_inr

subscriptions["arr_eur"] = subscriptions["arr_amount"] * usd_to_eur

subscriptions["arr_gbp"] = subscriptions["arr_amount"] * usd_to_gbp

In [35]:
def country_details(country):

    try:

        url = f"https://restcountries.com/v3.1/name/{country}"

        data = requests.get(url).json()[0]

        return pd.Series([
            data["capital"][0],
            data["region"],
            data["population"],
            list(data["currencies"].keys())[0],
            data["flags"]["png"]
        ])

    except:

        return pd.Series([None,None,None,None,None])

In [36]:
accounts[
[
"capital",
"region",
"population",
"currency",
"flag"
]
] = accounts["country"].apply(country_details)

In [42]:

subscriptions.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag,subscription_days,revenue_band,arr_inr,arr_eur,arr_gbp
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True,111.0,Enterprise,3.229923e+06,29298.601248,24997.77504
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True,NaN,High,9.657307e+05,8760.134544,7474.20912
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False,NaN,NaN,0.000000e+00,0.000000,0.00000
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True,20.0,High,1.153544e+06,10463.786160,8927.77680
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True,NaN,NaN,6.229137e+06,56504.445264,48209.99472


In [43]:
accounts.head()



,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag,customer_age_days,singup_month,singup_quarter,capital,region,population,currency,flag
0,A-2e4581,Company_0,Edtech,Us,2024-10-16,Partner,Basic,9,False,False,645,October,4,None,None,None,None,None
1,A-43a9e3,Company_1,Fintech,In,2023-08-17,Other,Basic,18,False,True,1071,August,3,None,None,None,None,None
2,A-0a282f,Company_2,Devtools,Us,2024-08-27,Organic,Basic,1,False,False,695,August,3,None,None,None,None,None
3,A-1f0ac7,Company_3,Healthtech,Uk,2023-08-27,Other,Basic,24,True,False,1061,August,3,None,None,None,None,None
4,A-ce550d,Company_4,Healthtech,Us,2024-10-27,Event,Enterprise,35,False,True,634,October,4,None,None,None,None,None


In [44]:
accounts.to_csv("accounts_api.csv", index=False)
subscriptions.to_csv("subscriptions_api.csv", index=False)
feature_usage.to_csv("feature_usage_api.csv", index=False)
support_tickets.to_csv("support_tickets_api.csv", index=False)
churn_events.to_csv("churn_events_api.csv", index=False)

In [45]:
from google.colab import files

files.download("accounts_api.csv")
files.download("subscriptions_api.csv")
files.download("feature_usage_api.csv")
files.download("support_tickets_api.csv")
files.download("churn_events_api.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>